# Sustainability Project Notebook

## Import Libraries and Read CSV Files

In [2]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import cvxpy as cp
from sklearn.covariance import LedoitWolf


In [ ]:
# each is_yyyy dataframe contains data for the monthly returns of the last 10 years. 
# is_2013 is the investment set used for building the 2014 portfolio as it contains returns from Jan. 2004 to Dec. 2013.

for year in range(2013, 2026):
    exec(f'is_{year} = pd.read_csv(f"processed_data/data_investment_set/investment_set_{year}.csv")')
    print(f'is_{year} loaded successfully.')



is_2013 loaded successfully.
is_2014 loaded successfully.
is_2015 loaded successfully.
is_2016 loaded successfully.
is_2017 loaded successfully.
is_2018 loaded successfully.
is_2019 loaded successfully.
is_2020 loaded successfully.
is_2021 loaded successfully.
is_2022 loaded successfully.
is_2023 loaded successfully.
is_2024 loaded successfully.
is_2025 loaded successfully.


## Creation of the Value-Weighted Portfolio

In [3]:
# Calculation of the weights for each investment set
for year in range(2013, 2026):
    df = globals()[f'is_{year}']

    # Creation of the new column 'MV_Y_weight'
    df['MV_Y_weight'] = df['MV_Y'] / df['MV_Y'].sum()

    # Relocation of 'MV_Y_weight' to the sixth position
    cols = list(df.columns)
    cols.remove('MV_Y_weight')
    cols.insert(5, 'MV_Y_weight')
    globals()[f'is_{year}'] = df[cols]

    # Sum of weights to verify that it equals 1
    print(f"Sum of weights for {year}: {df['MV_Y_weight'].sum()}")

is_2013

Sum of weights for 2013: 0.9999999999999999
Sum of weights for 2014: 1.0
Sum of weights for 2015: 1.0
Sum of weights for 2016: 1.0
Sum of weights for 2017: 1.0
Sum of weights for 2018: 1.0
Sum of weights for 2019: 1.0
Sum of weights for 2020: 0.9999999999999999
Sum of weights for 2021: 1.0
Sum of weights for 2022: 1.0
Sum of weights for 2023: 1.0
Sum of weights for 2024: 1.0000000000000002
Sum of weights for 2025: 1.0


/var/folders/b3/wvshm2ws24dc4b18w65382sh0000gn/T/ipykernel_44795/2492218673.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['MV_Y_weight'] = df['MV_Y'] / df['MV_Y'].sum()


,ISIN,NAME,Country,Region,MV_Y,MV_Y_weight,CO2_S1,CO2_S2,REV,Carbon_Intensity,...,2013-03-29 00:00:00,2013-04-30 00:00:00,2013-05-31 00:00:00,2013-06-28 00:00:00,2013-07-31 00:00:00,2013-08-30 00:00:00,2013-09-30 00:00:00,2013-10-31 00:00:00,2013-11-29 00:00:00,2013-12-31 00:00:00
0,AN8068571086,SLB,US,AMER,105861.90,0.005598,1790000.0,430000.0,42149000.0,0.0527,...,-0.0380,-0.0061,-0.0188,-0.0146,0.1349,-0.0010,0.0917,0.0607,-0.0566,0.0227
1,AT000000STR1,STRABAG SE,AT,EUR,3087.59,0.000163,968759.0,295141.0,17782013.0,0.0711,...,-0.0889,0.0266,-0.0669,-0.0270,0.0828,-0.0430,0.1655,0.0549,0.1161,-0.0004
2,AT0000606306,RAIFFEISEN BANK INTL.,AT,EUR,8185.61,0.000433,5161.0,43060.0,13350128.0,0.0036,...,-0.1008,0.0375,-0.0315,-0.1490,0.1007,0.1284,-0.0434,0.1239,0.0031,-0.0436
3,AT0000652011,ERSTE GROUP BANK,AT,EUR,13245.36,0.000700,32753.0,84281.0,16802146.0,0.0070,...,-0.1350,0.1245,0.0511,-0.1794,0.1433,0.0550,-0.0113,0.1167,-0.0010,-0.0106
4,AT0000720008,TELEKOM AUSTRIA,AT,EUR,3381.96,0.000179,27476.0,195246.0,5930020.0,0.0376,...,-0.0064,0.0442,-0.0014,-0.0686,0.0974,0.0339,0.1697,-0.0169,0.0330,-0.1099
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
798,US98389B1008,XCEL ENERGY,US,AMER,13656.38,0.000722,56535330.0,774475.0,10128223.0,5.6584,...,0.0446,0.0704,-0.0966,-0.0038,0.0568,-0.0678,-0.0009,0.0453,-0.0291,0.0071
799,US9839191015,XILINX DEAD - DELIST.15/02/22,US,AMER,9719.62,0.000514,1685.0,26545.0,2195457.0,0.0129,...,0.0241,-0.0068,0.0794,-0.0256,0.1787,-0.0650,0.0790,-0.0304,-0.0166,0.0335
800,US98421M1062,XEROX HOLDINGS,US,AMER,9815.17,0.000519,127000.0,186000.0,22390000.0,0.0140,...,0.0675,-0.0023,0.0245,0.0385,0.0695,0.0289,0.0367,-0.0340,0.1449,0.0744
801,US9884981013,YUM! BRANDS,US,AMER,29787.74,0.001575,156510.0,2377372.0,13633000.0,0.1859,...,0.0987,-0.0483,-0.0054,0.0235,0.0565,-0.0398,0.0196,-0.0475,0.1488,-0.0267


###  Calculating the Results of the 2014 Portfolio 

In [4]:
# Creation of new DataFrame with the weights and ISINs from is_2013
value_weighted_portfolio_2014 = is_2013[["ISIN", "NAME", "MV_Y_weight"]].copy()

# Selection of the 2014 returns columns (the last 12) and include 'ISIN' and 'Name'
columns_to_select = ['ISIN'] + list(is_2014.columns[-12:])
returns_2014 = is_2014[columns_to_select]

# Merging the 2014 returns with its Value Weighted Portfolio DataFrame on the 'ISIN' column
value_weighted_portfolio_2014 = value_weighted_portfolio_2014.merge(returns_2014, on='ISIN', how='left')

# 2014 Value Weighted Portfolio with returns
value_weighted_portfolio_2014


,ISIN,NAME,MV_Y_weight,2014-01-31 00:00:00,2014-02-28 00:00:00,2014-03-31 00:00:00,2014-04-30 00:00:00,2014-05-30 00:00:00,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-29 00:00:00,2014-09-30 00:00:00,2014-10-31 00:00:00,2014-11-28 00:00:00,2014-12-31 00:00:00
0,AN8068571086,SLB,0.005598,-0.0282,0.0667,0.0484,0.0415,0.0245,0.1381,-0.0811,0.0152,-0.0725,-0.0298,-0.1288,-0.0016
1,AT000000STR1,STRABAG SE,0.000163,0.0145,-0.0852,-0.0497,0.0782,0.0596,0.0626,-0.1331,0.0264,-0.1411,-0.0818,0.0652,-0.0462
2,AT0000606306,RAIFFEISEN BANK INTL.,0.000433,0.1367,-0.0926,-0.0436,-0.0542,0.0628,-0.0090,-0.1369,-0.0677,-0.1531,-0.0197,-0.0272,-0.2688
3,AT0000652011,ERSTE GROUP BANK,0.000700,0.0433,-0.0249,-0.0374,-0.0187,0.0455,-0.0703,-0.2033,-0.0015,-0.1095,0.1108,0.0670,-0.1428
4,AT0000720008,TELEKOM AUSTRIA,0.000179,0.1535,0.1139,0.0206,-0.0032,-0.0182,0.0114,-0.0213,-0.0197,-0.0390,-0.1687,-0.0086,-0.0233
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
798,US98389B1008,XCEL ENERGY,0.000722,0.0347,0.0477,0.0122,0.0497,-0.0348,0.0581,-0.0444,0.0406,-0.0424,0.1010,0.0140,0.0673
799,US9839191015,XILINX DEAD - DELIST.15/02/22,0.000514,0.0109,0.1307,0.0397,-0.1305,0.0012,0.0075,-0.1306,0.0344,0.0024,0.0503,0.0283,-0.0473
800,US98421M1062,XEROX HOLDINGS,0.000519,-0.1085,0.0129,0.0341,0.0699,0.0215,0.0124,0.0659,0.0415,-0.0374,0.0038,0.0512,-0.0027
801,US9884981013,YUM! BRANDS,0.001575,-0.1074,0.1032,0.0177,0.0261,0.0042,0.0503,-0.1415,0.0437,-0.0062,0.0040,0.0755,-0.0570


In [5]:
# Multiplying the returns of each stock by its corresponding weight and summing across all stocks to get the portfolio return for each month
value_weighted_portfolio_2014_returns = value_weighted_portfolio_2014.iloc[:, 3:].mul(value_weighted_portfolio_2014['MV_Y_weight'], axis=0).sum()
value_weighted_portfolio_2014_returns

2014-01-31 00:00:00   -0.036073
2014-02-28 00:00:00    0.054132
2014-03-31 00:00:00    0.007866
2014-04-30 00:00:00    0.018191
2014-05-30 00:00:00    0.015617
2014-06-30 00:00:00    0.011431
2014-07-31 00:00:00   -0.022088
2014-08-29 00:00:00    0.022326
2014-09-30 00:00:00   -0.025609
2014-10-31 00:00:00    0.000225
2014-11-28 00:00:00    0.024581
2014-12-31 00:00:00   -0.019277
dtype: float64

### Calculating the Results of Every Year by Creating a Function Based on the Above Code

In [6]:
def calculate_value_weighted_portfolio_returns(start_year, end_year):
    returns = {}

    for year in range(start_year, end_year + 1):
        try:
            prev_year = year - 1
            prev_df = globals()[f'is_{prev_year}']  # Get investment set DataFrame
            current_df = globals()[f'is_{year}']   # Get current year returns

            value_weighted_portfolio_yyyy = prev_df.loc[:, ("ISIN", "NAME", "MV_Y_weight")].copy()
            columns_to_select = ['ISIN'] + list(current_df.columns[-12:])
            returns_yyyy = current_df[columns_to_select]

            value_weighted_portfolio_yyyy = value_weighted_portfolio_yyyy.merge(returns_yyyy, on='ISIN', how='left')
            value_weighted_portfolio_yyyy_returns = value_weighted_portfolio_yyyy.iloc[:, 3:].mul(value_weighted_portfolio_yyyy['MV_Y_weight'], axis=0).sum()

            returns[year] = value_weighted_portfolio_yyyy_returns

        except Exception as e:
            print(f"Error processing year {year}: {e}")

    return returns

returns = calculate_value_weighted_portfolio_returns(2014, 2025)

In [7]:
# Creation of a DataFrame from the returns dictionary
def create_returns_dataframe(returns_dict):
    # Initialize empty DataFrame with years as index and months as columns
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
              'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    df = pd.DataFrame(index=sorted(returns_dict.keys()), columns=months)

    # Populate the DataFrame
    for year, returns in returns_dict.items():
        df.loc[year] = returns.values

    return df

returns_df = create_returns_dataframe(returns)
returns_df

,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
2014,-0.036073,0.054132,0.007866,0.018191,0.015617,0.011431,-0.022088,0.022326,-0.025609,0.000225,0.024581,-0.019277
2015,-0.023956,0.060384,-0.020122,0.028179,0.000398,-0.024719,0.018982,-0.063968,-0.035347,0.078765,-0.008175,-0.027718
2016,-0.055225,-0.00378,0.073652,0.018299,0.005943,-0.017208,0.041557,0.005956,0.00254,-0.020009,0.019818,0.034682
2017,0.014644,0.030306,0.014557,0.019408,0.027144,0.002424,0.021292,-0.001141,0.028424,0.013895,0.024999,0.013984
2018,0.047312,-0.04897,-0.016743,0.012743,0.000035,0.00234,0.039607,0.00527,0.006229,-0.061881,0.013307,-0.080812
2019,0.080454,0.034346,0.011335,0.041109,-0.066263,0.073223,0.002844,-0.025744,0.031693,0.023969,0.031224,0.035967
2020,-0.014171,-0.090218,-0.145032,0.110129,0.037685,0.026677,0.03554,0.057468,-0.041721,-0.03273,0.162987,0.040866
2021,-0.012173,0.036578,0.050018,0.048155,0.022251,0.004683,0.015492,0.019158,-0.038854,0.055974,-0.026212,0.06123
2022,-0.03684,-0.025298,0.024008,-0.072222,0.005096,-0.086407,0.083894,-0.046596,-0.094245,0.08124,0.075282,-0.043181
2023,0.07844,-0.022239,0.033727,0.025486,-0.017188,0.06092,0.032352,-0.026291,-0.046666,-0.023822,0.093163,0.050721


## Minimum-Variance Portfolio

### Computation of the Sample Covariance Matrix of 2004-2013 Returns

In [8]:
mv_is_2013 = is_2013.copy()
mv_is_2013 = mv_is_2013.set_index('ISIN')
mv_is_2013

,NAME,Country,Region,MV_Y,MV_Y_weight,CO2_S1,CO2_S2,REV,Carbon_Intensity,2004-01-30 00:00:00,...,2013-03-29 00:00:00,2013-04-30 00:00:00,2013-05-31 00:00:00,2013-06-28 00:00:00,2013-07-31 00:00:00,2013-08-30 00:00:00,2013-09-30 00:00:00,2013-10-31 00:00:00,2013-11-29 00:00:00,2013-12-31 00:00:00
ISIN,,,,,,,,,,,,,,,,,,,,,
AN8068571086,SLB,US,AMER,105861.90,0.005598,1790000.0,430000.0,42149000.0,0.0527,0.1181,...,-0.0380,-0.0061,-0.0188,-0.0146,0.1349,-0.0010,0.0917,0.0607,-0.0566,0.0227
AT000000STR1,STRABAG SE,AT,EUR,3087.59,0.000163,968759.0,295141.0,17782013.0,0.0711,NaN,...,-0.0889,0.0266,-0.0669,-0.0270,0.0828,-0.0430,0.1655,0.0549,0.1161,-0.0004
AT0000606306,RAIFFEISEN BANK INTL.,AT,EUR,8185.61,0.000433,5161.0,43060.0,13350128.0,0.0036,NaN,...,-0.1008,0.0375,-0.0315,-0.1490,0.1007,0.1284,-0.0434,0.1239,0.0031,-0.0436
AT0000652011,ERSTE GROUP BANK,AT,EUR,13245.36,0.000700,32753.0,84281.0,16802146.0,0.0070,0.0467,...,-0.1350,0.1245,0.0511,-0.1794,0.1433,0.0550,-0.0113,0.1167,-0.0010,-0.0106
AT0000720008,TELEKOM AUSTRIA,AT,EUR,3381.96,0.000179,27476.0,195246.0,5930020.0,0.0376,0.1377,...,-0.0064,0.0442,-0.0014,-0.0686,0.0974,0.0339,0.1697,-0.0169,0.0330,-0.1099
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
US98389B1008,XCEL ENERGY,US,AMER,13656.38,0.000722,56535330.0,774475.0,10128223.0,5.6584,0.0200,...,0.0446,0.0704,-0.0966,-0.0038,0.0568,-0.0678,-0.0009,0.0453,-0.0291,0.0071
US9839191015,XILINX DEAD - DELIST.15/02/22,US,AMER,9719.62,0.000514,1685.0,26545.0,2195457.0,0.0129,0.0847,...,0.0241,-0.0068,0.0794,-0.0256,0.1787,-0.0650,0.0790,-0.0304,-0.0166,0.0335
US98421M1062,XEROX HOLDINGS,US,AMER,9815.17,0.000519,127000.0,186000.0,22390000.0,0.0140,0.0609,...,0.0675,-0.0023,0.0245,0.0385,0.0695,0.0289,0.0367,-0.0340,0.1449,0.0744


In [9]:
# Extraction of the return columns 
return_columns_2013 = mv_is_2013.columns[9:]
returns_data_2013 = mv_is_2013[return_columns_2013]

# Transpose of the data to have firms as columns and dates as rows (allowing us to compute the covariance matrix between stocks instead of between months)
returns_2013 = returns_data_2013.T

returns_2013

ISIN,AN8068571086,AT000000STR1,AT0000606306,AT0000652011,AT0000720008,AT0000743059,AT0000746409,BE0003470755,BE0003565737,BE0003593044,...,US95040Q1040,US9581021055,US9621661043,US9633201069,US9694571004,US98389B1008,US9839191015,US98421M1062,US9884981013,US98956P1021
2004-01-30 00:00:00,0.1181,NaN,NaN,0.0467,0.1377,0.0677,0.0866,-0.0519,0.1674,0.0144,...,0.0918,-0.1323,-0.0334,0.0454,0.0326,0.0200,0.0847,0.0609,-0.0143,0.0867
2004-02-27 00:00:00,0.0572,NaN,NaN,0.0798,0.0294,0.0139,0.1748,0.0384,0.0476,0.0451,...,-0.0243,0.1134,0.0617,-0.0337,-0.0661,0.0087,0.0021,-0.0342,0.0920,-0.0112
2004-03-31 00:00:00,-0.0099,NaN,NaN,0.0717,-0.0068,0.1738,0.0288,-0.0488,0.0243,-0.0118,...,0.0752,-0.0140,0.0038,-0.0558,0.0116,0.0302,-0.1007,0.0304,0.0260,-0.0246
2004-04-30 00:00:00,-0.0833,NaN,NaN,-0.0001,0.0213,-0.0340,0.0851,0.0426,-0.0390,-0.0360,...,-0.1988,-0.2805,-0.0904,-0.0488,0.0763,-0.0606,-0.1099,-0.0783,0.0211,0.0822
2004-05-31 00:00:00,-0.0200,NaN,NaN,0.0448,-0.0486,-0.0151,0.0140,-0.0345,0.0432,-0.0187,...,0.0160,0.1312,0.0216,0.0223,0.1563,0.0155,0.0869,0.0082,-0.0332,0.0689
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-08-30 00:00:00,-0.0010,-0.0430,0.1284,0.0550,0.0339,0.0427,-0.0237,0.0281,0.0961,-0.0306,...,-0.0358,-0.0370,-0.0282,-0.0348,0.0606,-0.0678,-0.0650,0.0289,-0.0398,-0.0526
2013-09-30 00:00:00,0.0917,0.1655,-0.0434,-0.0113,0.1697,0.0727,0.1784,0.0797,0.1202,0.0589,...,0.0153,0.0266,0.0457,0.1383,0.0136,-0.0009,0.0790,0.0367,0.0196,0.0411
2013-10-31 00:00:00,0.0607,0.0549,0.1239,0.1167,-0.0169,-0.0330,0.0382,0.0446,0.1104,0.0318,...,0.0396,0.0983,0.0700,-0.0029,-0.0179,0.0453,-0.0304,-0.0340,-0.0475,0.0649
2013-11-29 00:00:00,-0.0566,0.1161,0.0031,-0.0010,0.0330,0.0273,-0.0741,-0.0263,0.0476,0.0217,...,-0.1259,0.0777,-0.0094,0.0508,-0.0137,-0.0291,-0.0166,0.1449,0.1488,0.0451


In [10]:
# Computation of the covariance matrix
covariance_matrix_2013 = returns_2013.cov()

covariance_matrix_2013

ISIN,AN8068571086,AT000000STR1,AT0000606306,AT0000652011,AT0000720008,AT0000743059,AT0000746409,BE0003470755,BE0003565737,BE0003593044,...,US95040Q1040,US9581021055,US9621661043,US9633201069,US9694571004,US98389B1008,US9839191015,US98421M1062,US9884981013,US98956P1021
ISIN,,,,,,,,,,,,,,,,,,,,,
AN8068571086,0.008909,0.007115,0.007107,0.006425,0.003050,0.005525,0.004201,0.004039,0.005657,0.002463,...,0.001698,0.004843,0.003522,0.003468,0.004854,0.000804,0.002970,0.003696,0.001743,0.002051
AT000000STR1,0.007115,0.020084,0.015876,0.018472,0.007008,0.009324,0.009518,0.008618,0.019284,0.005365,...,0.003925,0.009444,0.006253,0.011712,0.005125,0.002182,0.005327,0.007310,0.004718,0.005501
AT0000606306,0.007107,0.015876,0.020599,0.017884,0.005767,0.008832,0.008466,0.008683,0.018307,0.005102,...,0.003808,0.007212,0.007148,0.009357,0.005336,0.001601,0.003779,0.006548,0.003921,0.004742
AT0000652011,0.006425,0.018472,0.017884,0.020277,0.005071,0.007409,0.008306,0.009039,0.017912,0.004445,...,0.003693,0.007453,0.006864,0.010877,0.005052,0.001492,0.004394,0.006308,0.003554,0.004781
AT0000720008,0.003050,0.007008,0.005767,0.005071,0.007344,0.004317,0.004352,0.002930,0.005162,0.002249,...,0.001597,0.002434,0.001628,0.001477,0.001724,0.000872,0.002226,0.002584,0.001480,0.001552
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
US98389B1008,0.000804,0.002182,0.001601,0.001492,0.000872,0.000970,0.000691,0.000484,0.000796,0.000607,...,0.001249,0.000934,0.001278,0.000619,0.000705,0.001610,0.000707,0.000796,0.000673,0.000260
US9839191015,0.002970,0.005327,0.003779,0.004394,0.002226,0.002568,0.002415,0.002801,0.004932,0.001386,...,0.001528,0.003943,0.002873,0.004095,0.002401,0.000707,0.005934,0.003089,0.001767,0.002956
US98421M1062,0.003696,0.007310,0.006548,0.006308,0.002584,0.003327,0.002774,0.003834,0.006362,0.002913,...,0.002669,0.002971,0.004179,0.006479,0.003449,0.000796,0.003089,0.008420,0.002957,0.003276


### Computation of the Minimum-Variance Portfolio Weights for 2014 Using the Sample Covariance Matrix

In [11]:
# DIRECT METHOD
# Inverse of the covariance matrix
inv_cov_matrix_2013 = np.linalg.inv(covariance_matrix_2013.values)

# Vector of ones for the number of stocks in the covariance matrix
ones_2013 = np.ones(covariance_matrix_2013.shape[0])

# Calculate weights of the 2014 portfolio
# Computation of the formula numerator for the 2014 portfolio (Σ⁻¹ * 1)
numerator_2014 = inv_cov_matrix_2013 @ ones_2013

# Computation of the formula denominator for the 2014 portfolio (1ᵀ * Σ⁻¹ * 1)
denominator_2014 = ones_2013.T @ inv_cov_matrix_2013 @ ones_2013

# Calculating the weights
min_var_weights_2014_direct = numerator_2014 / denominator_2014

# Verifying the weights
print(min_var_weights_2014_direct.sum())
print(f"Positive values: {(min_var_weights_2014_direct > 0).sum()}")
print(f"Zero values: {(min_var_weights_2014_direct == 0).sum()}")
print(f"Negative values: {(min_var_weights_2014_direct < 0).sum()}")

if (min_var_weights_2014_direct < 0).sum() > 0:
    print("Negative weights found, indicating short selling.")
else: 
    print("No negative weights, no short selling -> optimal solution")
    
# (Displaying the weights)
# min_var_weights_2014_direct

1.0000000000000124
Positive values: 387
Zero values: 0
Negative values: 416
Negative weights found, indicating short selling.


In [12]:
# 1st ALGORITHMIC METHOD: SLSQP
n_assets_2013 = covariance_matrix_2013.shape[0]

# Initial guess (equal weights)
w0 = np.ones(n_assets_2013) / n_assets_2013

# Constraints: sum of weights = 1
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})

# Bounds: weights between 0 and 1 (no short selling)
bounds = tuple((0, 1) for _ in range(n_assets_2013))

# Objective function to minimize
def portfolio_variance(w):
    return w.T @ covariance_matrix_2013.values @ w

# Solving the optimization problem
result_min_var_slsqp_2014 = minimize(portfolio_variance, w0, method='SLSQP', bounds=bounds, constraints=constraints)

# Array of minimum variance weights
min_var_weights_2014_slsqp = result_min_var_slsqp_2014.x

# Verifying the weights
print(min_var_weights_2014_slsqp.sum())
print(f"Positive values: {(min_var_weights_2014_slsqp > 0).sum()}")
print(f"Zero values: {(min_var_weights_2014_slsqp == 0).sum()}")
print(f"Negative values: {(min_var_weights_2014_slsqp < 0).sum()}")

if (min_var_weights_2014_slsqp < 0).sum() > 0:
    print("Negative weights found, indicating short selling.")
else: 
    print("No negative weights, no short selling -> optimal solution")
    
# (Displaying the weights)
# min_var_weights_2014_slsqp

1.0000000000000004
Positive values: 424
Zero values: 379
Negative values: 0
No negative weights, no short selling -> optimal solution


In [13]:
# Scaling check for convergence of the SLSQP method

# Scaling factor
scaling_factor = 10000

# Scaled covariance matrix
scaled_covariance_matrix = scaling_factor * covariance_matrix_2013

# Objective function to minimize (scaled)
def portfolio_variance_scaled(w):
    return w.T @ scaled_covariance_matrix @ w

# Solving the optimization problem (scaled)
result_min_var_slsqp_2014_scaled = minimize(portfolio_variance_scaled, w0, method='SLSQP', bounds=bounds, constraints=constraints)

# Array of minimum variance weights (scaled)
min_var_weights_2014_slsqp_scaled = result_min_var_slsqp_2014_scaled.x

# Compare objective values
original_objective = result_min_var_slsqp_2014.fun
scaled_objective = result_min_var_slsqp_2014_scaled.fun
print("Original objective value:", original_objective)
print("Scaled objective value:", scaled_objective)
print("Expected scaled objective:", scaling_factor * original_objective)
difference = scaled_objective - scaling_factor * original_objective
print("Difference in objective values:", difference)

if abs(difference) < 1e-6:
    print("Objective values are consistent, indicating numerical stability.")
else:
    print("Objective values differ significantly, indicating potential numerical instability.")

Original objective value: 0.0003670405112515549
Scaled objective value: 3.559018618824787
Expected scaled objective: 3.670405112515549
Difference in objective values: -0.11138649369076203
Objective values differ significantly, indicating potential numerical instability.


### Computation of the Minimum-Variance Portfolio Weights for 2014 Using Ledoit-Wolf

In [14]:
# 2nd ALGORITHMIC METHOD: CVXPY(OSQP) with Ledoit–Wolf shrinkage estimator for covariance matrix
# Removing assets with NaN values to ensure Ledoit–Wolf can compute the covariance matrix

# Remove assets with NaNs in returns_2013
returns_2013_lw = returns_2013.dropna(axis=1)  # Drops columns (assets) with any NaN
print("Number of assets:", len(returns_2013_lw.columns))

# Fit Ledoit–Wolf to get a PSD covariance matrix
lw = LedoitWolf()
covariance_matrix_2013_lw = lw.fit(returns_2013_lw).covariance_

# Verify the matrix is PSD
eigvals = np.linalg.eigvalsh(covariance_matrix_2013_lw)
print("All eigenvalues >= 0:", np.all(eigvals >= 0))
print("Min eigenvalue:", eigvals.min())

# Get the number of assets and define the optimization variable
n_assets_2013_lw = covariance_matrix_2013_lw.shape[0]
w = cp.Variable(n_assets_2013_lw)

# cp.quad_form(w, covariance_matrix_2013_lw) calculates w' * Σ * w, the portfolio variance
objective = cp.Minimize(cp.quad_form(w, covariance_matrix_2013_lw))

# Define the constraints:
constraints = [cp.sum(w) == 1, w >= 0]

# Optimization problem with the objective and constraints
problem = cp.Problem(objective, constraints)

# Solve using the OSQP solver
problem.solve(solver=cp.OSQP)
min_var_weights_2014_lw = w.value

# Get the solver status
print("Solver status:", problem.status)

# Check weights sum to 1
print("Weights sum to 1:", np.sum(min_var_weights_2014_lw))

# Check no negatives (due to numerical noise)
print("Number of negative weights:", ((min_var_weights_2014_lw < 0).sum()))
print("Number of weights smaller than -1e-21:", ((min_var_weights_2014_lw < -1e-21).sum()))
if (min_var_weights_2014_lw < -1e-10).sum() > 0:
    print("Substantial negative weights found")
else: print("Set slightly negative weights to zero and renormalize")

print("Biggest weight:", np.max(min_var_weights_2014_lw))

Number of assets: 751
All eigenvalues >= 0: True
Min eigenvalue: 0.0011736215166327066
Solver status: optimal
Weights sum to 1: 1.0000000000000029
Number of negative weights: 58
Number of weights smaller than -1e-21: 0
Set slightly negative weights to zero and renormalize
Biggest weight: 0.08246969374905852


In [15]:
# Scaled problem
scaling_factor = 10000
scaled_covariance_matrix = scaling_factor * covariance_matrix_2013_lw

w_scaled = cp.Variable(n_assets_2013_lw)
objective_scaled = cp.Minimize(cp.quad_form(w_scaled, scaled_covariance_matrix))
constraints_scaled = [cp.sum(w_scaled) == 1, w_scaled >= 0]

problem_scaled = cp.Problem(objective_scaled, constraints_scaled)
problem_scaled.solve(solver=cp.OSQP)

min_var_weights_2014_lw_scaled = w_scaled.value
scaled_objective = problem_scaled.value

# Compare objective values
print("Original objective value:", problem.value)
print("Scaled objective value:", scaled_objective)
print("Expected scaled objective:", scaling_factor * problem.value)
difference = scaled_objective - scaling_factor * problem.value
print("Difference in objective values:", difference)

if abs(difference) < 1e-6:
    print("Objective values are consistent, indicating numerical stability.")
else:
    print("Objective values differ significantly, indicating potential numerical instability.")

Original objective value: 0.00038057495556926635
Scaled objective value: 3.8057495556926435
Expected scaled objective: 3.8057495556926635
Difference in objective values: -1.9984014443252818e-14
Objective values are consistent, indicating numerical stability.


In [16]:
# Clip negative weights to zero
min_var_weights_2014_lw = np.maximum(min_var_weights_2014_lw, 0)

# Re-normalize to ensure weights sum to 1
min_var_weights_2014_lw /= np.sum(min_var_weights_2014_lw)

# Verify the weights sum to 1
print("Weights sum to 1:", np.sum(min_var_weights_2014_lw))
# Verify no negative weights remain
print("Number of negative weights:", (min_var_weights_2014_lw < 0).sum())

# (Displaying the weights)
# min_var_weights_2014_lw

Weights sum to 1: 1.0
Number of negative weights: 0


### 2014 Results with both algorithmic methods

In [17]:
# Transpose the 2004-2013 lw returns dataframe to have assets as rows and dates as columns for calculations
# This dataframe only contains the returns of the assets that were kept for the Ledoit–Wolf method (i.e., those without NaN values)
# Name it "min_var_weights" for further use
min_var_weights_2013_lw = returns_2013_lw.T.copy() 

# Add the weights to the returns DataFrame
min_var_weights_2013_lw['min_var_weight'] = min_var_weights_2014_lw

# set the 'min_var_weight' column as the first column
min_var_weights_2013_lw = min_var_weights_2013_lw[['min_var_weight']]

# Get is_2014 with only the ISIN and last 12 columns (returns of 2014)
is_2014_returns = is_2014[['ISIN'] + list(is_2014.columns[-12:])]

# Merge both DataFrames on the ISIN column
min_var_portfolio_2014_lw_returns = min_var_weights_2013_lw.merge(is_2014_returns, left_index=True, right_on='ISIN', how='left')

# Set the ISIN as the index column
min_var_portfolio_2014_lw_returns = min_var_portfolio_2014_lw_returns.set_index('ISIN')

min_var_portfolio_2014_lw_returns

,min_var_weight,2014-01-31 00:00:00,2014-02-28 00:00:00,2014-03-31 00:00:00,2014-04-30 00:00:00,2014-05-30 00:00:00,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-29 00:00:00,2014-09-30 00:00:00,2014-10-31 00:00:00,2014-11-28 00:00:00,2014-12-31 00:00:00
ISIN,,,,,,,,,,,,,
AN8068571086,3.557755e-22,-0.0282,0.0667,0.0484,0.0415,0.0245,0.1381,-0.0811,0.0152,-0.0725,-0.0298,-0.1288,-0.0016
AT0000652011,2.131335e-21,0.0433,-0.0249,-0.0374,-0.0187,0.0455,-0.0703,-0.2033,-0.0015,-0.1095,0.1108,0.0670,-0.1428
AT0000720008,3.834458e-22,0.1535,0.1139,0.0206,-0.0032,-0.0182,0.0114,-0.0213,-0.0197,-0.0390,-0.1687,-0.0086,-0.0233
AT0000743059,4.772898e-22,-0.0971,0.0519,-0.0032,0.0299,-0.0751,0.0895,-0.1097,-0.0372,-0.1309,-0.0662,-0.0774,-0.0815
AT0000746409,1.114827e-21,-0.0135,0.0052,-0.0294,0.0072,0.0038,-0.0044,-0.0292,0.0447,0.0256,0.0031,-0.0152,-0.0698
...,...,...,...,...,...,...,...,...,...,...,...,...,...
US98389B1008,0.000000e+00,0.0347,0.0477,0.0122,0.0497,-0.0348,0.0581,-0.0444,0.0406,-0.0424,0.1010,0.0140,0.0673
US9839191015,3.184997e-22,0.0109,0.1307,0.0397,-0.1305,0.0012,0.0075,-0.1306,0.0344,0.0024,0.0503,0.0283,-0.0473
US98421M1062,9.881284e-22,-0.1085,0.0129,0.0341,0.0699,0.0215,0.0124,0.0659,0.0415,-0.0374,0.0038,0.0512,-0.0027


In [18]:
# Multiplying the returns of each stock by its corresponding weight and summing across all stocks to get the portfolio returns
min_var_portfolio_2014_lw_returns = min_var_portfolio_2014_lw_returns.iloc[:, 1:].mul(min_var_portfolio_2014_lw_returns['min_var_weight'], axis=0).sum()
min_var_portfolio_2014_lw_returns


2014-01-31 00:00:00   -0.004533
2014-02-28 00:00:00    0.046974
2014-03-31 00:00:00    0.005375
2014-04-30 00:00:00    0.019809
2014-05-30 00:00:00    0.008205
2014-06-30 00:00:00    0.010237
2014-07-31 00:00:00   -0.040123
2014-08-29 00:00:00    0.031067
2014-09-30 00:00:00   -0.038434
2014-10-31 00:00:00    0.046787
2014-11-28 00:00:00    0.029096
2014-12-31 00:00:00    0.006501
dtype: float64

In [19]:
# Creation of a DataFrame with the weights and ISINs from is_2013 for the SLSQP method
min_var_weights_2013_samplecov = is_2013[["ISIN"]].copy()

# Adding the SLSQP weights to the DataFrame
min_var_weights_2013_samplecov['min_var_weight'] = min_var_weights_2014_slsqp

# Get the ISINs and last 12 columns (returns of 2014) from is_2014
is_2014_returns = is_2014[['ISIN'] + list(is_2014.columns[-12:])]

# Merge both DataFrames on the ISIN column
min_var_portfolio_2014_slsqp_returns = min_var_weights_2013_samplecov.merge(is_2014_returns, on='ISIN', how='left')
# Set the ISIN as the index column
min_var_portfolio_2014_slsqp_returns = min_var_portfolio_2014_slsqp_returns.set_index('ISIN')

min_var_portfolio_2014_slsqp_returns

,min_var_weight,2014-01-31 00:00:00,2014-02-28 00:00:00,2014-03-31 00:00:00,2014-04-30 00:00:00,2014-05-30 00:00:00,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-29 00:00:00,2014-09-30 00:00:00,2014-10-31 00:00:00,2014-11-28 00:00:00,2014-12-31 00:00:00
ISIN,,,,,,,,,,,,,
AN8068571086,0.000000e+00,-0.0282,0.0667,0.0484,0.0415,0.0245,0.1381,-0.0811,0.0152,-0.0725,-0.0298,-0.1288,-0.0016
AT000000STR1,1.733160e-18,0.0145,-0.0852,-0.0497,0.0782,0.0596,0.0626,-0.1331,0.0264,-0.1411,-0.0818,0.0652,-0.0462
AT0000606306,0.000000e+00,0.1367,-0.0926,-0.0436,-0.0542,0.0628,-0.0090,-0.1369,-0.0677,-0.1531,-0.0197,-0.0272,-0.2688
AT0000652011,8.605859e-19,0.0433,-0.0249,-0.0374,-0.0187,0.0455,-0.0703,-0.2033,-0.0015,-0.1095,0.1108,0.0670,-0.1428
AT0000720008,1.098636e-18,0.1535,0.1139,0.0206,-0.0032,-0.0182,0.0114,-0.0213,-0.0197,-0.0390,-0.1687,-0.0086,-0.0233
...,...,...,...,...,...,...,...,...,...,...,...,...,...
US98389B1008,6.017023e-05,0.0347,0.0477,0.0122,0.0497,-0.0348,0.0581,-0.0444,0.0406,-0.0424,0.1010,0.0140,0.0673
US9839191015,4.918216e-19,0.0109,0.1307,0.0397,-0.1305,0.0012,0.0075,-0.1306,0.0344,0.0024,0.0503,0.0283,-0.0473
US98421M1062,2.111845e-18,-0.1085,0.0129,0.0341,0.0699,0.0215,0.0124,0.0659,0.0415,-0.0374,0.0038,0.0512,-0.0027


In [20]:
# Multiplying the returns of each stock by its corresponding weight and summing across all stocks to get the portfolio returns
min_var_portfolio_2014_slsqp_returns = min_var_portfolio_2014_slsqp_returns.iloc[:, 1:].mul(min_var_portfolio_2014_slsqp_returns['min_var_weight'], axis=0).sum()
min_var_portfolio_2014_slsqp_returns

2014-01-31 00:00:00   -0.000031
2014-02-28 00:00:00    0.050466
2014-03-31 00:00:00    0.006592
2014-04-30 00:00:00    0.021431
2014-05-30 00:00:00    0.007804
2014-06-30 00:00:00    0.005127
2014-07-31 00:00:00   -0.044340
2014-08-29 00:00:00    0.030179
2014-09-30 00:00:00   -0.044355
2014-10-31 00:00:00    0.046537
2014-11-28 00:00:00    0.032140
2014-12-31 00:00:00    0.009403
dtype: float64

### Function implementation

In [10]:
# Renaming the DataFrame
min_var_portfolio_2014 = min_var_2014.copy()

# Selection of the "ISIN" and last 13 columns from is_2014
is_2014_last_13 = is_2014[["ISIN"] + is_2014.columns[-13:].tolist()]

# Merging with min_var_portfolio on "ISIN"
min_var_portfolio_2014 = min_var_portfolio_2014.merge(
    is_2014_last_13,
    on="ISIN",
    how="left"
)

min_var_portfolio_2014


,ISIN,NAME,min_var_weights,2014-01-31 00:00:00,2014-02-28 00:00:00,2014-03-31 00:00:00,2014-04-30 00:00:00,2014-05-30 00:00:00,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-29 00:00:00,2014-09-30 00:00:00,2014-10-31 00:00:00,2014-11-28 00:00:00,2014-12-31 00:00:00,MV_Y_weights
0,AN8068571086,SLB,0.000000e+00,-0.0282,0.0667,0.0484,0.0415,0.0245,0.1381,-0.0811,0.0152,-0.0725,-0.0298,-0.1288,-0.0016,0.005275
1,AT000000STR1,STRABAG SE,3.856853e-18,0.0145,-0.0852,-0.0497,0.0782,0.0596,0.0626,-0.1331,0.0264,-0.1411,-0.0818,0.0652,-0.0462,0.000155
2,AT0000606306,RAIFFEISEN BANK INTL.,0.000000e+00,0.1367,-0.0926,-0.0436,-0.0542,0.0628,-0.0090,-0.1369,-0.0677,-0.1531,-0.0197,-0.0272,-0.2688,0.000524
3,AT0000652011,ERSTE GROUP BANK,0.000000e+00,0.0433,-0.0249,-0.0374,-0.0187,0.0455,-0.0703,-0.2033,-0.0015,-0.1095,0.1108,0.0670,-0.1428,0.000715
4,AT0000720008,TELEKOM AUSTRIA,1.167558e-18,0.1535,0.1139,0.0206,-0.0032,-0.0182,0.0114,-0.0213,-0.0197,-0.0390,-0.1687,-0.0086,-0.0233,0.000178
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
798,US98389B1008,XCEL ENERGY,3.334720e-04,0.0347,0.0477,0.0122,0.0497,-0.0348,0.0581,-0.0444,0.0406,-0.0424,0.1010,0.0140,0.0673,0.000667
799,US9839191015,XILINX DEAD - DELIST.15/02/22,5.702686e-19,0.0109,0.1307,0.0397,-0.1305,0.0012,0.0075,-0.1306,0.0344,0.0024,0.0503,0.0283,-0.0473,0.000563
800,US98421M1062,XEROX HOLDINGS,1.788617e-18,-0.1085,0.0129,0.0341,0.0699,0.0215,0.0124,0.0659,0.0415,-0.0374,0.0038,0.0512,-0.0027,0.000578
801,US9884981013,YUM! BRANDS,2.375484e-18,-0.1074,0.1032,0.0177,0.0261,0.0042,0.0503,-0.1415,0.0437,-0.0062,0.0040,0.0755,-0.0570,0.001380


In [11]:
# Selection of the columns with '2014' in their name
columns_2014 = min_var_portfolio_2014.filter(regex='2014', axis=1)

# Multiplication by the minimum variance weights
min_var_results_2014 = columns_2014.multiply(min_var_portfolio_2014['min_var_weights'], axis='index')

# Summing the weighted returns per column
min_var_results_2014 = min_var_results_2014.sum().to_frame().T

min_var_results_2014

,2014-01-31 00:00:00,2014-02-28 00:00:00,2014-03-31 00:00:00,2014-04-30 00:00:00,2014-05-30 00:00:00,2014-06-30 00:00:00,2014-07-31 00:00:00,2014-08-29 00:00:00,2014-09-30 00:00:00,2014-10-31 00:00:00,2014-11-28 00:00:00,2014-12-31 00:00:00
0,-0.000127,0.050459,0.006563,0.021482,0.007781,0.005169,-0.044305,0.03017,-0.044359,0.046456,0.032087,0.009405


In [12]:
# Based on the above code, we can create a function to compute the minimum variance portfolio results for every year
def compute_min_variance_portfolio(is_train, is_test):
    """
    Compute the minimum variance portfolio weights and results for the test year.

    Parameters:
    - is_train: DataFrame for the training year (e.g., 2013)
    - is_test: DataFrame for the test year (e.g., 2014)

    Returns:
    - DataFrame with the minimum variance portfolio results for the test year
    """
    # Extraction of the return columns 
    return_columns_train = is_train.columns[9:]
    returns_data_train = is_train[return_columns_train]

    # Transposition of the data to have firms as columns and dates as rows
    returns_train = returns_data_train.T

    # Computation of the covariance matrix
    covariance_matrix_train = returns_train.cov()
    n_assets_train = covariance_matrix_train.shape[0]

    # Initial guess (equal weights)
    w0 = np.ones(n_assets_train) / n_assets_train

    # Constraints: sum of weights = 1
    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})

    # Bounds: weights between 0 and 1 (no short selling, weights must be positive)
    bounds = tuple((0, 1) for _ in range(n_assets_train))

    # Objective function to minimize (portfolio variance)
    def portfolio_variance(w):
        return w.T @ covariance_matrix_train.values @ w

    # Solving the optimization problem
    result = minimize(portfolio_variance, w0, method='SLSQP', bounds=bounds, constraints=constraints)

    # Optimal weights
    min_var_weights = result.x

    # Selection of the first two columns from is_train
    weights_df = is_train.iloc[:, :2].copy()

    # Adding the array as a new column named "min_var_weights"
    weights_df['min_var_weights'] = min_var_weights

    # Renaming the dataframe
    min_var_portfolio = weights_df.copy()

    # Selection of the "ISIN" and last 13 columns from is_test
    is_test_last_13 = is_test[["ISIN"] + is_test.columns[-13:].tolist()]

    # Merge with min_var_portfolio on "ISIN"
    min_var_portfolio = min_var_portfolio.merge(
        is_test_last_13,
        on="ISIN",
        how="left"
    )

    # Multiplication of the returns by the weights and sum the results per column
    min_var_results = min_var_portfolio.iloc[:, 4:16].multiply(min_var_portfolio['min_var_weights'], axis='index')
    min_var_results = min_var_results.sum().to_frame().T

    return min_var_results

# Dictionary to store results for each year
min_var_results = {}

# Loop from 2014 to 2025
for year in range(2014, 2026):
    train_year = year - 1
    test_year = year

    # Getting the training and test DataFrames dynamically
    is_train = globals()[f'is_{train_year}']
    is_test = globals()[f'is_{test_year}']

    # Computing the minimum variance portfolio results
    min_var_results[f'min_var_results_{test_year}'] = compute_min_variance_portfolio(is_train, is_test)

# Now, min_var_results contains the results for each year from 2014 to 2025

In [13]:
# Initializing an empty list to hold each year's DataFrame
all_years_dfs = []

# Defining the standard column names
standard_columns = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Year']

# Loop through each year in the dictionary
for year in range(2014, 2026):
    key = f'min_var_results_{year}'
    if key in min_var_results:
        # Get the DataFrame for the current year
        year_df = min_var_results[key].copy()
        # Add the 'Year' column
        year_df['Year'] = year
        # Renaming the columns to standard names
        year_df.columns = standard_columns
        # Appending to the list
        all_years_dfs.append(year_df)

# Concatenating all yearly DataFrames into one big DataFrame
all_min_var_results = pd.concat(all_years_dfs, ignore_index=True)

# Setting 'Year' as the index
all_min_var_results.set_index('Year', inplace=True)
all_min_var_results

,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Year,,,,,,,,,,,,
2014,0.050459,0.006563,0.021482,0.007781,0.005169,-0.044305,0.030170,-0.044359,0.046456,0.032087,0.009405,0.001958
2015,0.006015,-0.021259,0.006875,0.007947,-0.016015,0.055187,-0.033228,0.002718,0.034358,-0.000112,0.023247,0.001853
2016,0.002787,0.056670,0.024581,0.013283,0.039859,0.024135,-0.052874,0.002031,-0.013149,-0.009819,0.019884,0.001764
2017,0.019381,-0.008005,0.037396,0.012934,-0.005676,0.020458,-0.040426,0.013192,-0.002014,0.029060,0.030151,0.001624
2018,-0.061701,-0.011342,0.004314,0.023433,0.030294,0.014698,0.020941,-0.016788,-0.012110,0.003952,-0.055982,0.001436
2019,0.048446,0.035867,0.017867,-0.016139,0.040462,-0.004746,-0.011433,0.006717,-0.015844,0.035339,0.061958,0.000930
2020,-0.091156,-0.146847,0.131446,0.033234,-0.004679,0.064433,0.045020,-0.019568,-0.013563,0.104142,0.040416,0.001196
2021,-0.003703,0.084050,0.020653,0.024926,-0.016565,0.026083,0.006671,-0.056497,0.031156,-0.032725,0.080071,0.000954
2022,-0.012074,0.044489,-0.042473,0.010105,-0.047390,0.032034,-0.032231,-0.073946,0.061216,0.066465,-0.026073,0.000799
